In [1]:
import pandas as pd
import numpy as np
import mysql.connector
from sqlalchemy import create_engine

print("Gọi thành công thư viện Pandas, Numpy và các thư viện MySQL!")

Gọi thành công thư viện Pandas, Numpy và các thư viện MySQL!


In [5]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine

# 1. Cấu hình thông tin (Điền mật khẩu của bạn vào)
config = {
    'host': 'localhost',      
    'user': 'root',           
    'password': '123456789',
    'database': 'uber_lyft_db' 
}

# 2. Tạo Database nếu chưa có
try:
    conn_server = mysql.connector.connect(host=config['host'], user=config['user'], password=config['password'])
    cursor = conn_server.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {config['database']}")
    conn_server.close()
    print("1. Kiểm tra/Tạo Database thành công!")
except Exception as e:
    print("Lỗi tạo Database:", e)

# 3. KẾT NỐI BẰNG PYMYSQL (Cách này cực kỳ ổn định)
# Chú ý: Đã đổi chữ 'mysqlconnector' thành 'pymysql'
db_url = f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}/{config['database']}"
engine = create_engine(db_url)
print("2. Kết nối Engine thành công!")

# 4. ĐỌC VÀ NẠP DỮ LIỆU VÀO MYSQL (Có cơ chế tự động bảo vệ đường ống)
print("3. Đang đọc file CSV...")
cab_data = pd.read_csv('cab_rides.csv')
weather_data = pd.read_csv('weather.csv')

print("4. Đang nạp dữ liệu vào MySQL (Vui lòng đợi 1-2 phút)...")
# Dùng with engine.begin() để đảm bảo nếu có lỗi, nó sẽ tự động Rollback (hủy) ngay
try:
    with engine.begin() as conn:
        cab_data.to_sql('rides', con=conn, if_exists='replace', index=False)
        weather_data.to_sql('weather', con=conn, if_exists='replace', index=False)
    print("=> THÀNH CÔNG! Đã nạp toàn bộ dữ liệu vào MySQL.")
except Exception as e:
    print("=> CÓ LỖI XẢY RA KHI NẠP (Đã tự động Rollback):", e)

1. Kiểm tra/Tạo Database thành công!
2. Kết nối Engine thành công!
3. Đang đọc file CSV...
4. Đang nạp dữ liệu vào MySQL (Vui lòng đợi 1-2 phút)...
=> THÀNH CÔNG! Đã nạp toàn bộ dữ liệu vào MySQL.


In [ ]:
# Câu lệnh SQL ghép 2 bảng và lọc dữ liệu lỗi
query = """
    SELECT 
        r.id, r.cab_type, r.name as ride_type, r.price, r.distance, 
        r.surge_multiplier, r.time_stamp, r.source, r.destination,
        w.temp, w.clouds, w.pressure, w.rain, w.humidity, w.wind
    FROM rides r
    LEFT JOIN weather w 
        ON r.source = w.location
    WHERE r.price IS NOT NULL
"""

print("Đang dùng lệnh SQL để ghép bảng và làm sạch...")
# Kéo dữ liệu từ SQL về lại Python, lưu vào biến df_clean
df_clean = pd.read_sql_query(query, con=engine)

print(f"Thành công! Thu được {df_clean.shape[0]} dòng dữ liệu hoàn chỉnh.")
display(df_clean.head())

Đang dùng lệnh SQL để ghép bảng và làm sạch...
